# <font color="littleblue">**Para correr el servidor en la nube correctamente, ejecuta en orden los siguientes bloques de codigo**

### <font color="yellow">***1***</font> ***Monta google Drive.***

#### *Asegurate de que tienes los archivos del modelo en la nube, como se indica en las instrucciones. En la ventana emergente que aparecerá cuando ejecute lo siguiente, simplemente conceda los permisos necesarios*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



### <font color="yellow">***2***</font> ***Instala streamlit y ngrok.***

#### *Solo ejecute la siguiente celda de codigo:*

In [2]:
# Instalar las herramientas necesarias
!pip install -q streamlit
!pip install -q pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.0 MB/s eta 0:00:00



### <font color="yellow">***3***</font> ***Creamos la interfaz del clasficador.***

#### *El mismo que utilizaremos para interactuar en el navegador, con el siguiente codigo:*

In [3]:
%%writefile app.py
import streamlit as st
import joblib

# 1. Configuración de la página
st.set_page_config(page_title="Ticket Classifier | Portal", page_icon="💼", layout="wide")

# --- INYECCIÓN DE CSS PARA DISEÑO ---
st.markdown("""
<style>
/* 1. Fondo principal: Borra de vino */
.stApp {
    background-color: #7F1D1D;
}

/* 2. FORZAR TODOS LOS TEXTOS A BLANCO */
h1, h2, h3, h4, h5, h6, p, .stMarkdown, label {
    color: #FFFFFF !important;
}

/* 3. Estilo para el título principal */
.main-title {
    background-color: #450A0A;
    color: #FFFFFF !important;
    text-align: center;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    font-weight: 700;
    padding: 20px;
    border-radius: 8px;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.4);
    margin-bottom: 25px;
    border-bottom: 2px solid #F8CBAD;
}

/* 4. Subtítulo dentro del banner */
.sub-title {
    font-size: 0.50em;
    font-weight: 500;
    color: #F8CBAD !important;
    display: block;
    margin-top: 8px;
    letter-spacing: 2px;
}

/* 5. TAMAÑO DE LAS MÉTRICAS (RESULTADOS) */
/* Achica el valor principal (Ej. "high", "Request") */
div[data-testid="stMetricValue"] > div, div[data-testid="stMetricValue"] {
    font-size: 24px !important;
    color: #FFFFFF !important;
}

/* Mantiene el tamaño original del subtítulo pero asegura que sea blanco */
div[data-testid="stMetricLabel"] * {
    color: #FFFFFF !important;
}

/* 6. FONDOS PARA NOTIFICACIONES Y ALERTAS */
/* Fondo amarillo clásico de advertencia para el Toast */
div[data-testid="stToast"] {
    background-color: #FFF3CD !important;
    border: 1px solid #FFEEBA !important;
    border-left: 5px solid #FFC107 !important;
    border-radius: 4px;
}

/* Forzar el texto del Toast a un color oscuro para que contraste con el fondo amarillo */
div[data-testid="stToast"] * {
    color: #856404 !important;
}

/* Mantenemos el resto de las alertas (st.info, st.error) con el color oscuro que definiste */
div[data-testid="stAlert"] {
    background-color: #450A0A !important;
    border: 1px solid #F8CBAD !important;
}
</style>
""", unsafe_allow_html=True)

# 2. Encabezado
st.markdown("""
<h1 class="main-title">
    Ticket Classifier
    <span class="sub-title">SISTEMA AUTOMATIZADO DE CLASIFICACIÓN DE TICKETS</span>
</h1>
""", unsafe_allow_html=True)

st.info("""
💡 **Instructions / Instrucciones:**

Paste the email content in the box below

Pega el contenido del correo electrónico en el cuadro de abajo.
""")

# 3. Rutas a los modelos en Google Drive
ruta_vectorizador = '/content/drive/MyDrive/Modelos_Tickets/vectorizador.pkl'
ruta_type = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_type_RF.pkl'
ruta_queue = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_queue_RF.pkl'
ruta_priority = '/content/drive/MyDrive/Modelos_Tickets/mejor_modelo_priority_RF.pkl'

# 4. Función para cargar modelos en caché
@st.cache_resource
def cargar_modelos():
    vectorizador = joblib.load(ruta_vectorizador)
    modelo_type = joblib.load(ruta_type)
    modelo_queue = joblib.load(ruta_queue)
    modelo_priority = joblib.load(ruta_priority)
    return vectorizador, modelo_type, modelo_queue, modelo_priority

try:
    vectorizador, modelo_type, modelo_queue, modelo_priority = cargar_modelos()
    st.toast("✅ Interface loaded successfully / Interfaz cargada exitosamente")
except Exception as e:
    st.error(f"❌ Error loading interface / Error al cargar la interfaz: {e}")

# 5. Interfaz de usuario (Entrada de texto)
st.markdown("### Input Ticket / Ingreso de Ticket")
ticket_input = st.text_area(
    "Cuerpo del correo:",
    height=200,
    placeholder="Example / Ejemplo: The main dashboard is down and I cannot work...",
    label_visibility="collapsed"
)

# 6. Botón centrado
col_btn1, col_btn2, col_btn3 = st.columns([1, 1, 1])
with col_btn2:
    btn_clasificar = st.button("📊 Run Analytics / Clasificar Ticket", type="primary", use_container_width=True)

# 7. Lógica de inferencia
if btn_clasificar:
    if ticket_input.strip():
        # Validación de longitud de caracteres
        if len(ticket_input.strip()) < 20:
            st.error("❌ **Error:** The text is too short. Please enter at least 20 characters / El texto es demasiado corto. Por favor ingresa al menos 20 caracteres.")
        else:
            with st.spinner("🧠 Processing / Procesando ..."):
                # A. Vectorización
                texto_vectorizado = vectorizador.transform([ticket_input])

                # B. Predicción
                pred_type = str(modelo_type.predict(texto_vectorizado)[0])
                pred_queue = str(modelo_queue.predict(texto_vectorizado)[0])
                pred_priority = str(modelo_priority.predict(texto_vectorizado)[0])

                # C. Presentación de resultados
                st.markdown("---")
                st.markdown("### Analysis Results / Resultados del Análisis")

                col1, col2, col3 = st.columns(3)
                with col1:
                    st.metric(label="⚡ Priority / Prioridad", value=pred_priority)
                with col2:
                    st.metric(label="🏷️ Type / Tipo", value=pred_type)
                with col3:
                    st.metric(label="📥 Queue / Departamento", value=pred_queue)

                st.markdown("###")

                # Mapeos en INGLÉS
                map_priority_en = {
                    "high": "addressed immediately",
                    "medium": "addressed promptly",
                    "low": "addressed following the regular support flow"
                }
                map_type_en = {
                    "incident": "an Incident",
                    "request": "a Request",
                    "problem": "a Problem",
                    "change": "a Change"
                }
                map_queue_en = {
                    "technical support": "Technical Support",
                    "product support": "Product Support",
                    "customer service": "Customer Service",
                    "it support": "IT Support",
                    "billing and payments": "Billing and Payments",
                    "returns and exchanges": "Returns and Exchanges",
                    "service outages and maintenance": "Service Outages and Maintenance",
                    "sales and pre-sales": "Sales and Pre-Sales",
                    "human resources": "Human Resources",
                    "general inquiry": "General Inquiries"
                }

                # Mapeos en ESPAÑOL
                map_priority_es = {
                    "high": "atendido de manera inmediata",
                    "medium": "atendido a la brevedad",
                    "low": "atendido siguiendo el flujo regular de soporte"
                }
                map_type_es = {
                    "incident": "un Incidente",
                    "request": "una Petición",
                    "problem": "un Problema",
                    "change": "un Cambio"
                }
                map_queue_es = {
                    "technical support": "Soporte Técnico",
                    "product support": "Soporte de Producto",
                    "customer service": "Servicio al Cliente",
                    "it support": "Soporte de TI",
                    "billing and payments": "Facturación y Pagos",
                    "returns and exchanges": "Devoluciones y Cambios",
                    "service outages and maintenance": "Interrupciones de servicio y Mantenimiento",
                    "sales and pre-sales": "Ventas y Preventas",
                    "human resources": "Recursos Humanos",
                    "general inquiry": "Consultas Generales"
                }

                # Traducciones seguras - INGLÉS
                en_priority = map_priority_en.get(pred_priority.lower().strip(), f"priority {pred_priority}")
                en_type = map_type_en.get(pred_type.lower().strip(), f"a {pred_type} ticket")
                en_queue = map_queue_en.get(pred_queue.lower().strip(), f"the {pred_queue} department")

                # Traducciones seguras - ESPAÑOL
                es_priority = map_priority_es.get(pred_priority.lower().strip(), f"prioridad {pred_priority}")
                es_type = map_type_es.get(pred_type.lower().strip(), f"un ticket de tipo {pred_type}")
                es_queue = map_queue_es.get(pred_queue.lower().strip(), f"el área de {pred_queue}")

                # --- ASIGNACIÓN DE ICONO SEMÁFORO ---
                prioridad_llave = pred_priority.lower().strip()
                if prioridad_llave == "high":
                    icono_semaforo = "🔴"
                elif prioridad_llave == "medium":
                    icono_semaforo = "🟡"
                elif prioridad_llave == "low":
                    icono_semaforo = "🟢"
                else:
                    icono_semaforo = "⚪"

                # Construcción del mensaje final
                mensaje_sistema = f"""
                 {icono_semaforo} The email should be **{en_priority}**, which corresponds to **{en_type}** associated with **{en_queue}**.

                 {icono_semaforo} El correo debe ser **{es_priority}**, corresponde con **{es_type}** asociada con **{es_queue}**.
                """
                st.markdown(mensaje_sistema)

    else:
        st.warning("⚠️ Please enter the ticket text / Por favor, ingresa el texto del correo antes de clasificar.")

Writing app.py



### <font color="yellow">***4***</font> ***Por ultimo creamos un enlace donde accederemos al clasificador.***
#### *Para utilizarlo simplemente haga click en el enlace que aparecerá a continuacion cuando haya terminado la ejecucion, por favor no desconecte este entorno hasta que hayas terminado su uso. De lo contrario el clasificador no funcionará.*


In [4]:
import subprocess
from pyngrok import ngrok
import time
import IPython # Importamos las herramientas interactivas de IPython

# 1. Configura tu Authtoken de ngrok
ngrok.set_auth_token("3Erks083VlGKcIxVal5m9oFKAMf_6QGnKXSvVamB1DHUxbaQw")

ngrok.kill()
get_ipython().system_raw('nohup streamlit run app.py &')
time.sleep(3)

public_url = ngrok.connect(8501)

# Extraemos el string de la URL
url = public_url.public_url

print(f"✅ Tu aplicación de Streamlit está corriendo en la nube, por favor visita el siguiente enlace: {url}")

# JavaScript para cargar automaticamente la interfaz en el navegador.
javascript_code = f"window.open('{url}', '_blank');"
display(IPython.display.Javascript(javascript_code))

✅ Tu aplicación de Streamlit está corriendo en la nube, por favor visita el siguiente enlace: https://surreal-gentleman-impotency.ngrok-free.dev


<IPython.core.display.Javascript object>